<a href="https://colab.research.google.com/github/myDream616828/CF_Mai23280018_Minh23280029/blob/main/%5BCF%5D_VWMA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install backtesting pandas numpy yfinance pmdarima arch pandas_ta

import yfinance as yf
import pandas as pd
import numpy as np
import pmdarima as pm
from arch import arch_model
from backtesting import Strategy, Backtest
from backtesting.lib import crossover

# --- THAM SỐ CẤU HÌNH ---
TICKER = 'TSLA'
START_DATE = '2015-01-01'
END_DATE = '2025-10-01'
CAPITAL = 100000.0
K= 1.5 # GIẢM LẠI K
MAX_RISK_PCT = 0.01 # Rủi ro tối đa/lệnh
df=pd.read_csv('TSLA.csv')

/usr/local/lib/python3.12/dist-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


In [2]:
def find_optimal_arima_params(series):
    # Dùng auto_arima để tìm p, d, q tối ưu
    model = pm.auto_arima(series,
                          start_p=1, start_q=1,
                          max_p=5, max_q=5,
                          d=None, # Cho phép tự động kiểm tra d
                          seasonal=False,
                          trace=False,
                          stepwise=True,
                          suppress_warnings=True,
                          error_action='ignore')
    return model.order

#Dự báo biến động Garch
def garch_rolling_forecast(returns_series, lookback_window=252*2):
    # Dùng list để lưu kết quả dự báo
    sigma_forecasts = [np.nan] * lookback_window

    #Áp dụng scaling factor để cải thiện việc ước tính mô hình GARCH
    scaling_factor = 100 # Thử nghiệm với các giá trị khác nhau nếu cần
    scaled_returns = returns_series * scaling_factor

    # Lặp qua dữ liệu để tính GARCH Rolling Forecast
    for i in range(lookback_window, len(returns_series)):
        window = scaled_returns.iloc[i - lookback_window : i]
        try:
            # Fit GARCH(1,1) trên Log Returns [cite: 13]
            am = arch_model(window, vol='Garch', p=1, q=1, mean='Constant', dist='normal')
            res = am.fit(disp='off')
            #Dự báo Độ lệch chuẩn (Sigma) cho ngày t+1 [cite: 12]
            forecasts = res.forecast(horizon=1)
            # Lấy giá trị đầu tiên của dự báo phương sai và lấy căn bậc hai
            sigma = np.sqrt(forecasts.variance.iloc[-1, 0])
            # Đảo ngược scaling cho sigma
            sigma /= scaling_factor
        except Exception:
            sigma = sigma_forecasts[-1] if sigma_forecasts[-1] is not np.nan else 0.01 # Dùng giá trị cũ hoặc default

        sigma_forecasts.append(sigma)

    return pd.Series(sigma_forecasts, index=returns_series.index)

In [3]:
#Tải Dữ liệu và Tính Toán Features
def prepare_data_with_features(ticker, start_date, end_date):
    # 1. Tải Dữ liệu
    data = pd.read_csv('TSLA.csv')
    if 'Date' in data.columns:
        data['Date'] = pd.to_datetime(data['Date'])
        data.set_index('Date', inplace=True)

    #Ưu tiên 'Adj Close' làm cột 'Close' chính
    if 'Adj Close' in data.columns:
        data['Close'] = data['Adj Close'] # Ghi đè hoặc tạo cột 'Close' bằng 'Adj Close'
        data = data.drop(columns=['Adj Close']) #Xóa cột 'Adj Close' ban đầu sau khi sử dụng
    elif 'Close' not in data.columns:
        raise ValueError("Không tìm thấy cột 'Close' hoặc 'Adj Close' trong dữ liệu.")

    #Sau khi xử lý, chọn các cột cần thiết, đảm bảo cột 'Close' là duy nhất
    required_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    # Chỉ chọn các cột thực sự tồn tại trong DataFrame
    existing_columns = [col for col in required_cols if col in data.columns]
    df = data[existing_columns].copy()
    #Kiểm tra xem cột 'Close' có tồn tại sau khi chọn hay không
    if 'Close' not in df.columns:
        raise ValueError("Cột 'Close' bị thiếu sau khi chuẩn bị và chọn dữ liệu.")

    #Tính toán Log Returns (để đạt Tính Dừng)
    df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

    #Tính toán GARCH Sigma (Biến động dự báo)
    #GARCH Sigma là Feature cho Risk Management
    df['GARCH_Sigma'] = garch_rolling_forecast(df['Log_Return'])

    #Tính toán ARIMA Signal (Đơn giản hóa cho Backtest)
    optimal_order = find_optimal_arima_params(df['Log_Return'].dropna().iloc[:252*2])
    print(f"Optimal ARIMA Order cho MSFT (trên data ban đầu): {optimal_order}")

    #Tạo ARIMA Signal
    #Vì thư viện backtesting.py không hỗ trợ fit phức tạp trong next()--> sử dụng EMA Crossover làm SIGNAL_FEATURE và dùng GARCH_Sigma cho POS_SIZE.
    # EMA Crossover mô phỏng Tín hiệu Xu hướng (Signal Generation)
    df['SMA_20'] = df['Close'].rolling(20).mean()
    df['SMA_50'] = df['Close'].rolling(50).mean()

    df['EMA_10'] = df['Close'].ewm(span=10, adjust=False).mean()
    df['EMA_30'] = df['Close'].ewm(span=30, adjust=False).mean()


    # Xử lý NaN lần cuối
    df.dropna(inplace=True)
    return df

#Hàm Tính toán RSI (Thêm vào phần Feature Engineering)
def calculate_rsi(df, period=14):
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    return df

# Chạy tiền xử lý
data = prepare_data_with_features(TICKER, START_DATE, END_DATE)

# Chạy tính toán RSI trên dữ liệu đã có
data = calculate_rsi(data)
data.dropna(inplace=True)

Optimal ARIMA Order cho MSFT (trên data ban đầu): (0, 0, 0)


In [6]:
import pandas_ta as ta

class VWMACGARCHStrategy(Strategy):
    risk_per_trade = MAX_RISK_PCT
    K = K
    sl_factor = 3.0

    def init(self):
        # Lấy các Features VWMA
        self.vwma_short = self.I(lambda x: x, self.data.VWMA_10)
        self.vwma_long = self.I(lambda x: x, self.data.VWMA_30)
        self.garch_sigma = self.I(lambda x: x, self.data.GARCH_Sigma)
        self.price = self.data.Close

    def next(self):
        current_price = self.data.Close[-1]
        sigma_t_plus_1 = self.garch_sigma[-1]
        current_equity = self.equity

        # 1. TÍNH KHOẢNG CÁCH CẮT LỖ ĐỘNG
        sl_distance = self.sl_factor * current_price * sigma_t_plus_1

        # 2. TÍNH KÍCH THƯỚC VỊ THẾ
        dynamic_risk_per_share = self.K * current_price * sigma_t_plus_1
        max_risk_per_trade = current_equity * self.risk_per_trade

        if dynamic_risk_per_share <= 0: return

        position_size = max_risk_per_trade / dynamic_risk_per_share
        target_size = int(position_size)

        # 3. TẠO TÍN HIỆU VÀ ĐẶT LỆNH (VWMAC Crossover)

        if crossover(self.vwma_short, self.vwma_long):
            # Tín hiệu MUA (Xu hướng tăng nhanh, được xác nhận bằng Khối lượng)
            sl_price = current_price - sl_distance

            if not self.position.is_long:
                self.position.close()
                self.buy(size=target_size, sl=sl_price)

        elif crossover(self.vwma_long, self.vwma_short):
            # Tín hiệu BÁN
            sl_price = current_price + sl_distance

            if not self.position.is_short:
                self.position.close()
                self.sell(size=abs(target_size), sl=sl_price)

# --- BẮT ĐẦU CHẠY BACKTEST MỚI ---
# Add VWMA calculations to data DataFrame
data['VWMA_10'] = ta.vwma(data['Close'], data['Volume'], length=10)
data['VWMA_30'] = ta.vwma(data['Close'], data['Volume'], length=30)
data.dropna(inplace=True)

backtest_rsi = Backtest(data,VWMACGARCHStrategy ,
                        cash=100000.0,
                        commission=.002,
                        exclusive_orders=True,
                        finalize_trades=True)

stats_rsi = backtest_rsi.run()

# Phân Tích Chỉ số
print("\n--- KẾT QUẢ BACKTEST TỔNG HỢP CHO TSLA (RSI Signal, K=2) ---")
print(f"Vốn Khởi điểm: $100000.00")
print(f"Vốn Cuối cùng: ${stats_rsi['Equity Final [$]']:.2f}")
print("----------------------------------------")
print(f"Return (Annual) [%]: {stats_rsi['Return (Ann.) [%]']:.2f}%")
print(f"Max. Drawdown [%]: {stats_rsi['Max. Drawdown [%]']:.2f}%")
print(f"Sharpe Ratio: {stats_rsi['Sharpe Ratio']:.2f}")
print(f"Win Rate [%]: {stats_rsi['Win Rate [%]']:.2f}%")
print(f"Tổng số lệnh (Trades): {stats_rsi['# Trades']}")


INITIAL_CAPITAL = CAPITAL
FINAL_EQUITY = stats_rsi['Equity Final [$]']
ANNUAL_RETURN = stats_rsi['Return (Ann.) [%]']
MAX_DRAWDOWN = stats_rsi['Max. Drawdown [%]']
SHARPE_RATIO = stats_rsi['Sharpe Ratio']
WIN_RATE = stats_rsi['Win Rate [%]']
TOTAL_TRADES = stats_rsi['# Trades']

summary_data = {
    'Ticker': [TICKER],
    'Strategy': ['RSI + GARCH (K=2)'],
    'Initial_Capital': [INITIAL_CAPITAL],
    'Final_Equity': [FINAL_EQUITY],
    'Annual_Return_Pct': [ANNUAL_RETURN],
    'Max_Drawdown_Pct': [MAX_DRAWDOWN],
    'Sharpe_Ratio': [SHARPE_RATIO],
    'Win_Rate_Pct': [WIN_RATE],
    'Total_Trades': [TOTAL_TRADES]
}
performance_summary_df = pd.DataFrame(summary_data)
performance_summary_df


Backtest.run:   0%|          | 0/3226 [00:00<?, ?bar/s]


--- KẾT QUẢ BACKTEST TỔNG HỢP CHO TSLA (RSI Signal, K=2) ---
Vốn Khởi điểm: $100000.00
Vốn Cuối cùng: $284488.46
----------------------------------------
Return (Annual) [%]: 8.51%
Max. Drawdown [%]: -25.17%
Sharpe Ratio: 0.64
Win Rate [%]: 43.10%
Tổng số lệnh (Trades): 116


,Ticker,Strategy,Initial_Capital,Final_Equity,Annual_Return_Pct,Max_Drawdown_Pct,Sharpe_Ratio,Win_Rate_Pct,Total_Trades
0,TSLA,RSI + GARCH (K=2),100000.0,284488.458503,8.507164,-25.173375,0.638451,43.103448,116
